In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [6]:
df = spark.read.json("/home/jovyan/notebooks/Zadanie2/cwiczenia/RTA/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
!find /home/jovyan -name "transactions_10k.jsonl"

/home/jovyan/notebooks/Zadanie2/cwiczenia/RTA/transactions_10k.jsonl


In [7]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [8]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [9]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [11]:
from pyspark.sql import functions as F

df.groupBy("category") \
  .agg(
      F.sum("amount").alias("suma_PLN"),
      F.min("amount").alias("min_PLN"),
      F.max("amount").alias("max_PLN")
  ) \
  .orderBy("category") \
  .show()

+-----------+------------------+-------+-------+
|   category|          suma_PLN|min_PLN|max_PLN|
+-----------+------------------+-------+-------+
|elektronika|1520770.6899999995|    9.0| 9999.0|
|    książki| 851382.0799999991|    5.0|9107.25|
|     odzież| 849877.5500000007|    5.0|9696.63|
|    żywność| 789514.4300000003|    5.0|6916.92|
+-----------+------------------+-------+-------+



In [12]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [13]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [14]:
from pyspark.sql import functions as F

result = (
    df
    .groupBy(
        F.window("timestamp", "30 minutes"),
        "store"
    )
    .agg(
        F.count("tx_id").alias("liczba_tx"),
        F.round(F.sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        F.col("window.start").alias("od"),
        F.col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN"
    )
    .orderBy("od", "store")
)

result.show(truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [17]:
from pyspark.sql import functions as F
from pyspark.sql.functions import desc

krakow_top_hour = (
    df
    .filter(F.col("store") == "Kraków")
    .groupBy(F.window("timestamp", "1 hour"))
    .agg(
        F.round(F.sum("amount"), 2).alias("suma_PLN"),
        F.count("tx_id").alias("liczba_tx")
    )
    .select(
        F.col("window.start").alias("od"),
        F.col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN"
    )
    .orderBy(desc("suma_PLN"))
)

krakow_top_hour.show(truncate=False)

+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169     |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|821      |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|532      |201259.26|
+-------------------+-------------------+---------+---------+



In [18]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [20]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: Sliding ma więcej wierszy, ponieważ okna nachodzą na siebie.
# Każda transakcja może należeć do kilku okien, podczas gdy w tumbling każde zdarzenie trafia tylko do jednego okna.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") agreguje dane tylko po sklepie, czyli liczy wszystkie transakcje
# dla danego sklepu łącznie, bez uwzględnienia czasu.
# Natomiast groupBy(window(...), "store") agreguje dane zarówno po sklepie,
# jak i po przedziałach czasowych (oknach), dzięki czemu wyniki są liczone
# osobno dla każdego sklepu w każdym oknie czasowym.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna

In [21]:
# Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
# ODPOWIEDŹ: godz. 8:00
from pyspark.sql import functions as F

gdansk_min_avg = (
    df
    .filter(F.col("store") == "Gdańsk")
    .groupBy(F.window("timestamp", "1 hour"))
    .agg(
        F.round(F.avg("amount"), 2).alias("srednia_PLN"),
        F.count("tx_id").alias("liczba_tx")
    )
    .select(
        F.col("window.start").alias("od"),
        F.col("window.end").alias("do"),
        "liczba_tx",
        "srednia_PLN"
    )
    .orderBy("srednia_PLN")
)

gdansk_min_avg.show(1, truncate=False)

+-------------------+-------------------+---------+-----------+
|od                 |do                 |liczba_tx|srednia_PLN|
+-------------------+-------------------+---------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|766      |395.01     |
+-------------------+-------------------+---------+-----------+
only showing top 1 row



In [26]:
# Policz ile transakcji per kategoria było w oknie 09:00–09:30.
# ODPOWIEDŹ: elektronika = 611, książki: 622, odzież: 605, żywność: 567
tx_9_930 = (
    df
    .filter(
        (F.date_format(F.col("timestamp"), "HH:mm:ss") >= "09:00:00") &
        (F.date_format(F.col("timestamp"), "HH:mm:ss") < "09:30:00")
    )
    .groupBy("category")
    .agg(F.count("tx_id").alias("liczba_tx"))
    .orderBy("category")
)

tx_9_930.show()

+-----------+---------+
|   category|liczba_tx|
+-----------+---------+
|elektronika|      611|
|    książki|      622|
|     odzież|      605|
|    żywność|      567|
+-----------+---------+



In [28]:
# Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).
# ODPOWIEDŹ: godz. 9:15-9:30
peak_15min = (
    df
    .groupBy(F.window("timestamp", "15 minutes"))
    .agg(F.count("tx_id").alias("liczba_tx"))
    .select(
        F.col("window.start").alias("od"),
        F.col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(F.desc("liczba_tx"))
)

peak_15min.show(1, truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+
only showing top 1 row

